In [16]:
import torch
import numpy as np
from tqdm import tqdm
from nltk.stem import PorterStemmer
import json
import pickle

stemmer = PorterStemmer()

In [17]:
embed_dim = 300
top_k =10

In [18]:
with open('data/hotpotqa_vocab_term_to_id.json') as f:
    term_to_id = json.load(f)
with open('data/hotpotqa_vocab_id_to_term.json') as f:
    id_to_term = json.load(f)

In [19]:
vocab = list(term_to_id.keys())
len(vocab)

45042

In [20]:
def load_glove(path):
    embeddings_index = {}
    with open(path, encoding="utf8") as f:
        for i, line in tqdm(enumerate(f)):
            values = line.strip().split()
            word = " ".join(values[:-embed_dim])
            vector = np.asarray(values[-embed_dim:], dtype="float32")
            vector = torch.from_numpy(vector)
            embeddings_index[word] = vector
    print(f"Loaded {len(embeddings_index)} word vectors from GloVe.")
    return embeddings_index

glove_path = "glove.2024.wikigiga.300d.txt"
glove_vectors = load_glove(glove_path)

1291147it [01:21, 15856.94it/s]


Loaded 1291147 word vectors from GloVe.


In [21]:
def build_stem_to_words_glove(glove_dict):
    """Build mapping: stem → set of GloVe words"""
    stem_to_words_glove = {}
    for word in tqdm(glove_dict.keys()):
        stem = stemmer.stem(word)
        stem_to_words_glove.setdefault(stem, set()).add(word)
    return stem_to_words_glove

In [22]:
def compute_stem_avg_vectors(stem_to_words_glove, glove_dict, vocab_stems):
    """Precompute average GloVe vector for each stem"""
    stem_avg_vec = {}
    for stem in tqdm(vocab_stems):
        words = stem_to_words_glove.get(stem, set())
        vectors = [glove_dict[w] for w in words if w in glove_dict]
        if vectors:  # skip stems with no vectors
            stem_avg_vec[stem] = np.mean(vectors, axis=0)
    return stem_avg_vec

In [23]:
def normalize_matrix(mat):
    """Normalize rows of a matrix to unit length."""
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    return mat / (norms + 1e-8)

In [24]:
def top_k_similar_stems_batch(stem_avg_vec, top_k=10):
    stems = list(stem_avg_vec.keys())
    vecs = np.stack([stem_avg_vec[s] for s in stems])
    vecs = normalize_matrix(vecs)  # row-wise normalization

    # cosine similarity = dot product of normalized vectors
    sims_matrix = np.dot(vecs, vecs.T)  # shape: (num_stems, num_stems)

    expansion_map = {}
    for i, stem in enumerate(tqdm(stems)):
        sims = sims_matrix[i].copy()
        sims[i] = -np.inf  # exclude self
        top_indices = np.argsort(-sims)[:top_k]  # top-k indices
        top_stems = {stems[j] for j in top_indices}
        expansion_map[stem] = top_stems

    return expansion_map

In [25]:
stem_to_words_glove = build_stem_to_words_glove(glove_vectors)

100%|██████████| 1291147/1291147 [00:13<00:00, 97622.73it/s] 


In [26]:
stem_avg_vec = compute_stem_avg_vectors(stem_to_words_glove, glove_vectors, vocab)

100%|██████████| 45042/45042 [00:01<00:00, 39013.58it/s]


In [27]:
expansion_map = top_k_similar_stems_batch(stem_avg_vec, top_k=top_k)

100%|██████████| 44183/44183 [00:41<00:00, 1063.78it/s]


In [28]:
all_related_vocab_ids = set()
for stem in tqdm(vocab):
    related_stems = expansion_map.get(stem, set())
    stem_id = term_to_id[stem]
    for rel_stem in related_stems:
        related_stem_id = term_to_id[rel_stem]
        if related_stem_id > stem_id:
            tuple_to_be_added = (stem_id, related_stem_id)
            tuple_to_be_added_reversed = (related_stem_id, stem_id)
        else:
            tuple_to_be_added = (related_stem_id, stem_id)
            tuple_to_be_added_reversed = (stem_id, related_stem_id)
        if tuple_to_be_added not in all_related_vocab_ids:
            all_related_vocab_ids.add(tuple_to_be_added)
        if tuple_to_be_added_reversed not in all_related_vocab_ids:
            all_related_vocab_ids.add(tuple_to_be_added_reversed)
    if (stem_id, stem_id) not in all_related_vocab_ids:
        all_related_vocab_ids.add((stem_id, stem_id))

100%|██████████| 45042/45042 [00:00<00:00, 83000.43it/s]


In [29]:
len(all_related_vocab_ids)

738210

In [30]:
with open('data/glove_pairs.pkl', 'wb') as f:
    pickle.dump(all_related_vocab_ids, f)